# Tune quick guide with autolog enabled

## Introduction
In this notebook, you'll see how to perform Hyperparameter tuning tasks with FLAML for different scenarios. We'll have mlflow autolog enabled, so you'll also see how Hyperparameter tuning integrated with mlflow.

The scenarios are as below:

1. Tune a pyspark ml type model

    In this scenario, we'll tune a SynapseML lightGBM model which is pyspark ml type. For the mlflow integration, we'll not set mlflow experiment name and run name; the experiment name will be the notebook name by default, while the run names will be randomly generated words.

2. Tune a non spark model

    In this scenario, we'll tune an original lightGBM model which is sklearn type, i.e., non spark type. And we'll customize mlflow experiment name and run name.

Please ref [FLAML doc](https://microsoft.github.io/FLAML/docs/Getting-Started/) for more details of FLAML usage.

## Prerequisites
We need to install flaml for performing automl tasks.

In [ ]:
%pip install "flaml[synapse]@https://automlsaeastus.blob.core.windows.net/releases/FLAML-latest-py3-none-any.whl"

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, -1, Finished, Available)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.0/264.0 kB 1.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 3.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.9/212.9 kB 27.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 39.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 39.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 25.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.2/147.2 kB 37.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 41.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0 -> 23.1.1
[notice] To update, run: /nfs4/pyenv-68d60e80-db28-466f-bf73-59acbd00bbff/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



## Case 1. Tune a pyspark ml type model

In this scenario, we'll tune a SynapseML lightGBM model which is pyspark ml type. For the mlflow integration, we'll not set mlflow experiment name and run name; the experiment name will be the notebook name by default, while the run names will be randomly generated words.

![image-alt-text](https://synapseaisolutionsa.blob.core.windows.net/public/demo-images/tune_autolog_on_1.png)


In [ ]:
import mlflow
import flaml
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

import pyspark
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from synapse.ml.lightgbm import LightGBMRegressor

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, 9, Finished, Available)

In [ ]:
# Set pyspark autologging logModelAllowlist to include SynapseML models
spark.sparkContext._conf.set(
    "spark.mlflow.pysparkml.autolog.logModelAllowlistFile",
    "https://mmlspark.blob.core.windows.net/publicwasb/log_model_allowlist.txt",
)

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, 10, Finished, Available)

### Prepare Dataset

In [ ]:
data = fetch_california_housing()

feature_cols = ["f" + str(i) for i in range(data.data.shape[1])]
header = ["target"] + feature_cols
df = spark.createDataFrame(pd.DataFrame(data=np.column_stack((data.target, data.data)), columns=header)).repartition(1)
print("Dataframe has {} rows".format(df.count()))

# Convert features into a single vector column
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = featurizer.transform(df)["target", "features"]

train_data, test_data = data.randomSplit([0.85, 0.15], seed=41)

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, 11, Finished, Available)

/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:604: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.


### Define training function

In [ ]:
def train(config):
    """
    This train() function:
     - takes hyperparameters config as inputs (for tuning later)
     - returns the R^2 score on the test dataset

    Wrapping code as a function makes it easier to reuse the code later with FLAML.
    """
    lgr = LightGBMRegressor(
        objective="quantile",
        alpha=config["alpha"],
        learningRate=config["learningRate"],
        numLeaves=config["numLeaves"],
        labelCol="target",
        numIterations=config["numIterations"],
    )
    model = lgr.fit(train_data)
    # Define an evaluation metric and evaluate the model on the test dataset.
    predictions = model.transform(test_data)
    evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="target", metricName="r2")
    eval_metric = evaluator.evaluate(predictions)

    return {"r2": eval_metric}

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, 12, Finished, Available)

### Define search space

In [ ]:
params = {
    "alpha": flaml.tune.uniform(0, 1),
    "learningRate": flaml.tune.uniform(0.001, 1),
    "numLeaves": flaml.tune.randint(30, 100),
    "numIterations": flaml.tune.randint(100, 300),
}

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, 13, Finished, Available)

### Run tuning

In [ ]:
# no need to set use_spark since a spark model itself will run in parallel
analysis = flaml.tune.run(
    train,
    params,
    metric="r2",
    mode="max",
    num_samples=5,
)

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, 14, Finished, Available)

[flaml.tune.tune: 04-25 09:39:11] {530} INFO - Using search algorithm BlendSearch.
No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune
You passed a `space` parameter to OptunaSearch that contained unresolved search space definitions. OptunaSearch should however be instantiated with fully configured search spaces only. To use Ray Tune's automatic search space conversion, pass the space definition as part of the `config` argument to `tune.run()` instead.
[flaml.tune.tune: 04-25 09:39:11] {809} INFO - trial 1 config: {'alpha': 0.09743207287894917, 'learningRate': 0.64761881525086, 'numLeaves': 30, 'numIterations': 172}
[flaml.tune.tune: 04-25 09:39:27] {809} INFO - trial 2 config: {'alpha': 0.771320643266746, 'learningRate': 0.021731197410042098, 'numLeaves': 74, 'numI

In [ ]:
synapselgb_config = analysis.best_config
synapselgb_r2 = analysis.best_result['r2']
print(f"Best config: {synapselgb_config}")
print(f"R^2: {synapselgb_r2}")

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, 15, Finished, Available)

Best config: {'alpha': 0.5940316589938806, 'learningRate': 0.22926504794631342, 'numLeaves': 35, 'numIterations': 279}
R^2: 0.8094330941991653



## Case 2. Tune a non spark model

In this scenario, we'll tune an original lightGBM model which is sklearn type, i.e., non spark type. And we'll customize mlflow experiment name and run name.

![image-alt-text](https://synapseaisolutionsa.blob.core.windows.net/public/demo-images/tune_exp_1.png)


### Prepare Dataset

In [ ]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.15)

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, 28, Finished, Available)

### Define training function

In [ ]:
def train_lgb(config):
    lgr = LGBMRegressor(
        objective="quantile",
        alpha=config["alpha"],
        learningRate=config["learningRate"],
        numLeaves=config["numLeaves"],
        labelCol="target",
        numIterations=config["numIterations"],
    )
    model = lgr.fit(train_x, train_y, eval_metric=["l2"], eval_set=[(train_x, train_y)])
    # Define an evaluation metric and evaluate the model on the test dataset.
    pred_y = model.predict(test_x)
    r2 = r2_score(test_y, pred_y)

    return {"r2": r2}

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, 29, Finished, Available)

### Run tuning

In [ ]:
mlflow.set_experiment("tune_exp")  # customize the experiment name
with mlflow.start_run(nested=True, run_name="tune_run"):  # customize the run name
    analysis = flaml.tune.run(
        train_lgb,
        params,
        metric="r2",
        mode="max",
        num_samples=5,
        use_spark=True,  # use spark to parallelize the training
        n_concurrent_trials=2,
    )

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, 30, Finished, Available)

[flaml.tune.tune: 04-25 10:04:22] {530} INFO - Using search algorithm BlendSearch.
No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune
You passed a `space` parameter to OptunaSearch that contained unresolved search space definitions. OptunaSearch should however be instantiated with fully configured search spaces only. To use Ray Tune's automatic search space conversion, pass the space definition as part of the `config` argument to `tune.run()` instead.
[flaml.tune.tune: 04-25 10:04:22] {719} INFO - Number of trials: 2/5, 2 RUNNING, 0 TERMINATED
[Parallel(n_jobs=2)]: Using backend SparkDistributedBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:   10.0s
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:   12.5s remaining:    0.

In [ ]:
lgb_config = analysis.best_config
lgb_r2 = analysis.best_result['r2']
print(f"Best config: {lgb_config}")
print(f"R^2: {lgb_r2}")

StatementMeta(, 9d5a6101-80f3-41b5-baba-9b02e2422ece, 31, Finished, Available)

Best config: {'alpha': 0.5940316589938806, 'learningRate': 0.22926504794631342, 'numLeaves': 35, 'numIterations': 279}
R^2: 0.8184426499256436
